# Session 1: from a catalogue to a defensible result

Whether you work with survey catalogues or simulation snapshots, a defensible analysis keeps columns aligned, tracks physical units from provenance, tests assumptions explicitly, and separates data-quality cuts from physical selection effects.

Our 400-row galaxy table is **synthetic teaching data**: input distances $d$ are exact, and recession velocities follow an injected linear relation ($v = 70\,d + 300\text{ km/s}$) with independent Gaussian measurement noise. It is not a cosmological measurement.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable

# Locate the course root directory (no need to edit)
_candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / 'jupyter_data_languages']
for _candidate in _candidates:
    if (_candidate / 'course2026/__init__.py').is_file() and (_candidate / 'data/mock_galaxies.ecsv').is_file():
        sys.path.insert(0, str(_candidate))
        break
import course2026
ROOT = Path(course2026.__file__).resolve().parents[1]
plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11})

## Inspect the catalogue

Astropy's `QTable` stores tabular columns as unit-aware `Quantity` arrays so physical dimensions travel with the numbers. We load `mock_galaxies.ecsv`, an **Extended CSV (`.ecsv`)** file: plain-text CSV preceded by a `#`-commented YAML header that records column units, data types and provenance metadata — convenient for sharing and version-controlling both observational sub-catalogues and synthetic simulation tables.

Before running the next cell, predict what columns, units and metadata dictionary (`table.meta`) a survey or simulation table needs to record.

In [ ]:
table = QTable.read(ROOT / 'data/mock_galaxies.ecsv')
print('Rows:', len(table), '| metadata:', table.meta)
table[:3].pprint_all()

## Arrays, shapes and masks

Never apply separate row filters to individual columns: build **one shared boolean mask**, inspect how many rows it removes (`~valid` is boolean NOT), and apply it to every column so objects remain aligned. Calling `.to_value(u.Mpc)` converts compatible Astropy units and extracts a plain `float` NumPy array. Because observed flux dims with distance as $F \propto L / (4\pi d^2)$, astronomy's logarithmic `apparent_magnitude` scale runs backwards: **larger numbers mean fainter (typically more distant) galaxies**.

![Diagram: One mask preserves row alignment](../book/diagrams/shared-mask.svg)

**Keep objects aligned.** Filtering columns separately can produce matching array lengths while silently pairing one galaxy's distance with another's velocity. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/shared-mask.dot).

In [ ]:
distance = table['distance'].to_value(u.Mpc)
velocity = table['velocity'].to_value(u.km / u.s)
sigma = table['velocity_error'].to_value(u.km / u.s)
magnitude = np.asarray(table['apparent_magnitude'])

valid = (
    np.isfinite(distance) & np.isfinite(velocity) & np.isfinite(sigma)
    & np.isfinite(magnitude) & (distance > 0) & (sigma > 0)
)
print('Shape:', distance.shape, '| usable:', valid.sum(), '| removed:', (~valid).sum())
assert valid.sum() == 395
assert distance[valid].shape == velocity[valid].shape == sigma[valid].shape

### Task A

1. Count missing (`~np.isfinite`) velocities and nonpositive (`<= 0` or non-finite) velocity errors separately, and inspect the excluded rows with `table[~valid]`.
2. Explain why replacing missing velocities with `0.0` would corrupt a mean or linear fit *(hint: `np.count_nonzero(condition)` counts `True` entries).*

In [ ]:
# Task A: count missing velocities and nonpositive errors separately

### Shapes and broadcasting

NumPy aligns array shapes **from right to left (trailing axes first)**: two dimensions are compatible only when they are equal or one of them is `1`. Multiplying an `(N, 3)` multi-band table by a 1D per-object array of shape `(N,)` fails because trailing axis `3 != N`, whereas indexing with `[:, None]` reshapes `(N,)` to `(N, 1)` so one factor scales each object's three bands cleanly.

![Diagram: Broadcast one scale per object](../book/diagrams/broadcasting.svg)

**Scale rows, not columns.** Reshaping `(N,)` to `(N, 1)` stretches each object's correction factor across its three band columns. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/broadcasting.dot).

In [ ]:
# Synthetic 3-band fluxes (g, r, i) for the 395 usable galaxies: shape (395, 3)
flux_bands = (1e4 / distance[valid, None] ** 2) * np.array([0.8, 1.0, 1.3])
extinction_corr = np.linspace(0.9, 1.1, valid.sum())  # shape (395,)
corrected_flux = flux_bands * extinction_corr[:, None]

print('3-band matrix:', flux_bands.shape, '| row factor:', extinction_corr[:, None].shape)
assert corrected_flux.shape == (395, 3)
np.testing.assert_allclose(corrected_flux[-1], flux_bands[-1] * 1.1)

## Units and FITS

Never infer units from the size of a number: rely on documented headers and unit conversions. Astropy converts compatible units (`km/s` to `m/s`, preventing silent factor-of-1000 errors) and raises `UnitConversionError` when dimensions are physically incompatible.

In [ ]:
velocity_ms = table['velocity'].to(u.m / u.s)
np.testing.assert_allclose(velocity_ms.value[valid], 1000 * velocity[valid])
try:
    table['distance'].to(u.s)
except u.UnitConversionError:
    print('Expected: a distance cannot be converted to seconds')

Large astronomical catalogues and images are commonly archived in **FITS (Flexible Image Transport System)**, a binary format organised into numbered Header/Data Units (HDUs). Writing a table creates an empty Primary Header (`HDU 0`) followed by a Binary Table (`HDU 1`) whose `TUNITn` header cards preserve physical column units. Because standard FITS headers enforce strict 8-character keyword rules and 80-character card limits — unlike ECSV's free-form YAML header — we clear long free-text entries from `subset.meta` before writing.

![Diagram: ECSV text metadata vs FITS Header and Data Units](../book/diagrams/fits-vs-ecsv.svg)

**Text ECSV vs binary FITS.** Both formats preserve column units (`TUNITn` in FITS), while ECSV accommodates richer free-form dictionary metadata. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/fits-vs-ecsv.dot).

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / 'mock_subset.fits'
    subset = table[['distance', 'velocity', 'velocity_error']][valid].copy()
    subset.meta.clear()
    subset.write(path)
    restored = QTable.read(path)
    np.testing.assert_allclose(restored['distance'].to_value(u.Mpc), distance[valid])
    print('FITS round-trip rows:', len(restored), '| unit:', restored['distance'].unit)

### Task B

Explain to a partner which provenance metadata `subset.meta.clear()` discarded, and why `restored['distance'].to_value(u.Mpc)` is safer than `.value` if a collaborator saved distances in `kpc`. Write one `assert` check you would run immediately after reading an external FITS catalogue.

**Break** *(after completing Task B).*

In [ ]:
# Task B: write one assertion you would run after reading a FITS table

## Plot the data and its uncertainty

A scientific figure needs physical axis labels with units, readable uncertainties, and an explicit description of what subset is shown. Plotting every eighth usable galaxy keeps individual $1\sigma$ Gaussian velocity error bars legible; distances carry no horizontal error bars here because our synthetic inputs treat $d$ as exact.

In [ ]:
fig, ax = plt.subplots()
indices = np.flatnonzero(valid)[::8]
ax.errorbar(
    distance[indices], velocity[indices], yerr=sigma[indices],
    fmt='o', ms=3.5, alpha=0.8, capsize=2, label='Every 8th usable mock galaxy',
)
ax.set(
    xlabel='Distance (exact input) [Mpc]',
    ylabel='Velocity [km/s]',
    title='Synthetic catalogue: velocity relation and 1-sigma uncertainty',
)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

### Your turn

Copy and adapt the plot above to either (a) display the exact plotted and usable sample sizes (`n=... of ...`) in the legend or title, or (b) overplot the unselected faint galaxies (`valid & (magnitude >= 16.5)`) in light grey behind the bright sample.

In [ ]:
# Your turn: include sample counts (n=... of ...) or overplot faint usable galaxies in light grey

## Investigate a survey selection

Keep data-quality filtering (`valid`, 395 rows) distinct from physical or observational selection (`apparent_magnitude < 16.5`, 262 rows), and compare populations on **identical histogram bin edges**. Raw counts show where objects drop out; normalised probability densities compare distribution shapes.

In **observational** terms, `m < 16.5` is a flux-limited cut that preferentially loses distant galaxies (and in surveys with distance scatter, produces classical Malmquist bias). In **statistical/theoretical** terms, conditioning on $m < 16.5$ truncates the marginal distance distribution $p(d)$ — shifting $\bar{d}$ from $110.7\text{ Mpc}$ to $87.5\text{ Mpc}$ — while leaving the conditional relation $E[v \mid d]$ unbiased here because $d$ is exact and velocity noise is independent of $m$.

![Diagram: Filtering and survey selection are separate decisions](../book/diagrams/catalogue-selection.svg)

**Data hygiene vs survey cut.** Changing `magnitude_limit` alters the selected sample (262 of 395 at `m < 16.5`) without changing which 395 rows are valid. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/catalogue-selection.dot).

In [ ]:
magnitude_limit = 16.5
selected = valid & (magnitude < magnitude_limit)
print('Selected:', selected.sum(), 'of', valid.sum(), 'usable rows')
assert selected.sum() == 262

bins = np.linspace(20, 200, 13)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
titles = ['Counts per bin', 'Normalised probability density']
for ax, density, title in zip(axes, [False, True], titles):
    ax.hist(distance[valid], bins=bins, histtype='step', lw=2, density=density,
            label='All usable (n=395)')
    ax.hist(distance[selected], bins=bins, histtype='step', lw=2, density=density,
            label=f'm < {magnitude_limit} (n={selected.sum()})')
    ax.set(
        title=title,
        xlabel='Distance [Mpc]',
        ylabel='Probability density [1/Mpc]' if density else 'Objects per bin',
    )
axes[0].legend(fontsize=8.5)
fig.tight_layout()
plt.show()
print('Mean distances [Mpc]:', distance[valid].mean().round(1), distance[selected].mean().round(1))

### Task C

1. Using a separate variable `test_limit`, predict whether tightening the cut to `15.5` or relaxing it to `17.5` lowers or raises the selected count and mean distance, then verify both numerically.
2. Draft a three-sentence caption stating what is plotted, what rows were excluded, and why shifting $p(d)$ does **not** automatically bias $v(d)$ under this mock's noise model — then name one realistic effect that *would* bias the fit.

In [ ]:
# Task C: check counts and mean distances for test_limit = 15.5 and 17.5 (leave `selected` unchanged)
test_limit = 15.5

## Exit check

Before moving on, verify you can explain: why 395 of 400 rows are usable, why `(N, 3) * (N, 1)` broadcasts per row, why units must come from headers rather than guesswork, and why a flux limit changes $p(d)$ without changing data validity. Run **Restart Kernel and Run All Cells** to confirm clean top-to-bottom execution, then compare with [solutions](../book/solutions.md).